In [1]:
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))
print("project root:", root)


project root: e:\Projects\biomed-ml


In [2]:
import numpy as np
import pandas as pd
from biomed.datasets import BonnEEGDataset
from biomed.features import StatsFeatures, BandPowerFeatures, CombinedFeatures
from biomed.models import MODELS
from biomed.evaluation import cross_validate

X, y, groups = BonnEEGDataset().load()
F = CombinedFeatures([StatsFeatures(), BandPowerFeatures()]).transform(X)
target = (y == 1).astype(int)
print(F.shape, target.mean())


(11500, 21) 0.2


In [3]:
summary = {}
for name, factory in MODELS.items():
    res = cross_validate(factory, F, target, groups)
    summary[name] = res.drop(columns="fold").mean()
    print("done:", name)

pd.DataFrame(summary).T.round(3)


done: logreg
done: svm
done: random_forest


d:\ANACONDA\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "d:\ANACONDA\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "d:\ANACONDA\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ANACONDA\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass

done: hist_gb


,accuracy,precision,recall,f1,roc_auc
logreg,0.969,0.897,0.956,0.926,0.994
svm,0.973,0.905,0.970,0.936,0.994
random_forest,0.977,0.954,0.929,0.941,0.996
hist_gb,0.977,0.949,0.935,0.942,0.996


In [4]:
grouped = cross_validate(MODELS["random_forest"], F, target, groups, grouped=True)
naive = cross_validate(MODELS["random_forest"], F, target, groups, grouped=False)

pd.DataFrame({
    "grouped (correct)": grouped.drop(columns="fold").mean(),
    "naive split (leaky)": naive.drop(columns="fold").mean(),
}).round(4)


,grouped (correct),naive split (leaky)
accuracy,0.9768,0.9828
precision,0.9541,0.9603
recall,0.9291,0.9535
f1,0.9413,0.9568
roc_auc,0.9959,0.9978
